# 🦜 VieNeu-Audio (Colab)

Phiên bản này dùng backbone GGUF lượng tử hóa + codec ONNX (chạy CPU, không cần GPU). Runtime thường (không cần đổi sang GPU) là đủ.

Chạy lần lượt từng cell. Nếu mất kết nối giữa chừng, xem mục cuối cùng.

## 1. Gắn Google Drive
Code và output đều lưu ở đây — không mất khi mất kết nối.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/VieNeu-Audio'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print('Đã có sẵn:', os.listdir(PROJECT_DIR))

## 2. Upload code
Chỉ cần lần đầu, hoặc khi có bản code mới. Nếu `pipeline/` đã có sẵn ở Bước 1, bỏ qua cell này.

In [ ]:
from google.colab import files
import zipfile, io

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as zf:
    zf.extractall(PROJECT_DIR)
print(os.listdir(PROJECT_DIR))

## 3. Cài dependencies
ffmpeg, font tiếng Việt, vieneu, gradio.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto
!fc-cache -f
!pip install -q vieneu gradio soundfile "numpy<2.1" "requests==2.32.4"

## 4. Kiểm tra nhanh
Cần thấy `h264_nvenc`/`libx264` (encoder) và ít nhất 1 dòng font "Noto Sans". Thiếu font → phụ đề tiếng Việt sẽ lỗi thành ô vuông.

In [ ]:
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E "nvenc|qsv|libx264"
!fc-list | grep -i "noto sans" | head -3

## 5. Khởi chạy
In ra link `https://xxxxx.gradio.live` — mở link đó để dùng: **① Chọn giọng → ② Nghe mẫu → ③ Render → Video (Batch)**.

An toàn để chạy lại cell này nhiều lần (miễn Bước 1 đã chạy trong phiên hiện tại).

In [ ]:
import sys

try:
    PROJECT_DIR
except NameError:
    raise RuntimeError("Chạy lại Bước 1 (Gắn Drive) trước — runtime vừa được cấp phát lại.")

try:
    app.close()
except Exception:
    pass
for mod in list(sys.modules):
    if mod == "pipeline" or mod.startswith("pipeline."):
        del sys.modules[mod]
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from pipeline.auto_tts import app
app.queue().launch(share=True, debug=True)

## 🔁 Mất kết nối?

Bình thường trên Colab free (tab rảnh ~90 phút, hoặc phiên quá ~12 tiếng) — không mất phần đã render, chỉ cần chạy lại:

1. Connect lại.
2. Chạy lại Bước 1 (bắt buộc — máy ảo mới hoàn toàn).
3. Bỏ qua Bước 2 (code đã ở trong Drive).
4. Chạy lại Bước 3, 5.
5. Mở link Gradio mới, upload lại đúng các file đang xử lý dở, chạy tiếp — chương/phần đã xong sẽ tự bỏ qua.